# 🏦 NUST Bank - Qwen 3.5 4B Fine-Tuning Notebook

Fine-tune Qwen 3.5 4B on NUST Bank product knowledge using Unsloth + LoRA.

**Requirements:** GPU with CUDA 12.8, PyTorch 2.7.0+cu128

## Step 1: Install Dependencies

Install unsloth compatible with **torch 2.7.0 + CUDA 12.8**.

In [ ]:
%%capture
# Step 1a: Install unsloth with correct CUDA 12.8 + torch 2.7.0 support
!pip install --upgrade pip
!pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install "unsloth[cu128-torch270] @ git+https://github.com/unslothai/unsloth.git" --no-build-isolation

# Step 1b: Install remaining dependencies (--no-deps to avoid overwriting torch)
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
!pip install --upgrade "transformers>=5.2.0"

In [1]:
# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version:    {torch.version.cuda}")
print(f"GPU available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:        {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")

PyTorch version: 2.7.0+cu128
CUDA version:    12.8
GPU available:   True
GPU name:        NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory:      8.0 GB


## Step 2: Load Model with Unsloth + LoRA

In [1]:
from unsloth import FastLanguageModel
import torch
import os

# Fix for CUDA memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Keep seq length manageable for GPU VRAM
max_seq_length = 768

print("Downloading Qwen 3.5 4B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-4B",
    max_seq_length = max_seq_length,
    dtype = None,              # Auto-detect best dtype
    load_in_4bit = True,       # 4-bit quantization to save VRAM
    trust_remote_code = True,
)

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Keeps VRAM low
    random_state = 3407,
)
print("✅ Model loaded and LoRA applied!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\Awais Nazir\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0405 20:32:11.740000 27504 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_5 does not support SDPA - switching to fast eager.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:03<00:00, 230.79it/s]


Applying LoRA adapters...
Unsloth: Making `model.base_model.model.model.language_model` require gradients
✅ Model loaded and LoRA applied!


## Step 3: Prepare Training Data

In [2]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Apply the Qwen 2.5 chat template (compatible with Qwen 3.5)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# 2. Load training data from JSONL file
dataset = load_dataset("json", data_files="finetuning_data_chat_fixed.jsonl", split="train")

# 3. Format the data for training
def formatting_prompts_func(examples):
    texts = []
    for convo in examples["messages"]:
        # Ensure every message content is a plain string (not a list)
        for msg in convo:
            if isinstance(msg["content"], list):
                msg["content"] = msg["content"][0]["text"]

        # Apply template with enable_thinking=False for direct banking answers
        text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
            enable_thinking = False,
        )
        texts.append(text)

    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"✅ Success! Loaded {len(dataset)} rows of banking data.")
print("\n--- PREVIEW OF FORMATTED DATA ---")
print(dataset[0]["text"][:500])

Generating train split: 310 examples [00:00, 40990.99 examples/s]
Map:   0%|          | 0/310 [00:00<?, ? examples/s]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.

✅ Success! Loaded 310 rows of banking data.

--- PREVIEW OF FORMATTED DATA ---
<|im_start|>system
You are a helpful customer support assistant for NUST Bank. Answer the customer's question accurately and concisely using the bank's product knowledge.<|im_end|>
<|im_start|>user
I would like to open an account with my son, do u have any product for kids?<|im_end|>
<|im_start|>assistant
Yes our product is Little Champs Account. It is designed specifically for minors (individuals below the age of 18 years). A child requires the help of a parental/legal guardian to open this acc


## Step 4: Configure Trainer

In [3]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,   # Effective batch size = 4
        warmup_steps = 10,
        num_train_epochs = 5,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        max_grad_norm = 0.3,
    ),
)
print("✅ Trainer configured!")

Unsloth: Tokenizing ["text"]: 100%|██████████| 310/310 [00:00<00:00, 4217.08 examples/s]

✅ Trainer configured!


## Step 5: Train the Model

In [4]:
# Check VRAM before starting
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | Max Memory: {max_memory} GB")
print(f"VRAM Reserved (Thanks to 4-bit!): {start_gpu_memory} GB\n")

print("🚀 Starting Training...")
trainer_stats = trainer.train()

print(f"\n✅ Training Complete! Time taken: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes.")
print(f"Final training loss: {trainer_stats.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


GPU: NVIDIA GeForce RTX 4060 Laptop GPU | Max Memory: 7.996 GB
VRAM Reserved (Thanks to 4-bit!): 3.164 GB

🚀 Starting Training...


Step,Training Loss
1,11.035351
2,11.860162
3,11.216662
4,10.721810
5,10.396201
6,9.299001
7,9.441170
8,7.202450
9,6.236806
10,6.608104


KeyboardInterrupt: 

## Step 6: Save the LoRA Adapter

In [6]:
# Save the LoRA adapter weights
adapter_name = "qwen3.5_banking_lora"
model.save_pretrained(adapter_name)
tokenizer.save_pretrained(adapter_name)
print(f"💾 Adapter saved to folder: '{adapter_name}'!")

💾 Adapter saved to folder: 'qwen3.5_banking_lora'!


## Step 7: Test the Fine-Tuned Model

**Key fix:** Use plain string content (not list-of-dicts) and use the text tokenizer to avoid vision processor issues with Qwen 3.5.

In [7]:
from unsloth import FastLanguageModel
import torch

# 1. Enable 2x faster inference
FastLanguageModel.for_inference(model)

# 2. Grab the hidden text-only tokenizer to bypass all vision logic safely
if hasattr(tokenizer, "tokenizer"):
    text_tokenizer = tokenizer.tokenizer
else:
    text_tokenizer = tokenizer

# 3. Manually build the prompt using Qwen chat format
#    Using plain strings — NOT [{"type": "text", "text": "..."}] lists!
prompt = """<|im_start|>system
You are a helpful customer support assistant for NUST Bank. Answer the customer's question accurately and concisely using the bank's product knowledge.<|im_end|>
<|im_start|>user
What other Value added features does the Little Champs Account have?<|im_end|>
<|im_start|>assistant
"""

# 4. Tokenize with the text-only tokenizer (avoids vision processor errors)
inputs = text_tokenizer(prompt, return_tensors="pt").to("cuda")

print("\n--- QWEN 3.5 RESPONSE (Fine-Tuned) ---")

# 5. Generate with repetition penalty + sampling to avoid loops
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 256,
    use_cache = True,
    pad_token_id = text_tokenizer.eos_token_id,
    do_sample = True,
    temperature = 0.3,          # Low temp = more deterministic, less hallucination
    top_p = 0.9,
    repetition_penalty = 1.15,  # Penalizes repeated tokens — prevents loops
)

# 6. Decode only the new generated tokens (skip the prompt)
prompt_length = inputs.input_ids.shape[1]
response = text_tokenizer.batch_decode(outputs[:, prompt_length:], skip_special_tokens=True)[0]

print(response)



--- QWEN 3.5 RESPONSE (Fine-Tuned) ---
· Free First Cheque Book (Unlimited) *
· SMS Alerts on digital transactions
· E-statements facility
· Inter Branch Online Cash Withdrawal/ Deposit (Online)
· Internal Fund Transfer within NUST via branch (Online Transfer)
· I Net Banking & Mobile banking facilities for account holders above 18 years


In [8]:
# Test with multiple questions
test_questions = [
    "What is the Eligibility Criteria for NAA?",
    "Does NUST Bank offer a Digital Banking Account for Freelancers?",
    "What are the benefits of NUST Bank Auto Finance?",
    "What documents are required to open a Little Champs Account?",
    "What is the Target Market for NUST Special Deposit Account (ASDA)?",
]

for q in test_questions:
    prompt = f"""<|im_start|>system
You are a helpful customer support assistant for NUST Bank. Answer the customer's question accurately and concisely using the bank's product knowledge.<|im_end|>
<|im_start|>user
{q}<|im_end|>
<|im_start|>assistant
"""
    inputs = text_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 256,
        use_cache = True,
        pad_token_id = text_tokenizer.eos_token_id,
        do_sample = True,
        temperature = 0.3,
        top_p = 0.9,
        repetition_penalty = 1.15,
    )
    prompt_length = inputs.input_ids.shape[1]
    response = text_tokenizer.batch_decode(outputs[:, prompt_length:], skip_special_tokens=True)[0]
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response}")



Q: What is the Eligibility Criteria for NAA?
A: Resident Pakistani individuals, including women, who do not maintain any other account with NUST Bank are eligible to open this account.

Q: Does NUST Bank offer a Digital Banking Account for Freelancers?
A: Yes, Freelancer Digital Account is specially designed to meet the banking needs of freelancers & sole proprietors offering digital services.

Q: What are the benefits of NUST Bank Auto Finance?
A: · Attractive financing rates starting from 10% p.a — Profit Payment System (PPS)
· Minimum age to avail loan is 21 years, however age not to exceed 65 at the time of application
· Loan tenure can be extended up to maximum 8 years or as per prevailing SOC/BOQ
· Application will be processed within 3 working days subject to complete application with required documents

Q: What documents are required to open a Little Champs Account?
A: • A photocopy of front page of CNIC of minor’s legal guardian (Parent/Guardian)
• Any one valid ID proof of l

## Step 8: Download the Adapter (for Colab)

In [ ]:
# Zip and download (only works in Google Colab)
import os
if 'google.colab' in str(get_ipython()):
    !zip -r qwen3.5_banking_lora.zip qwen3.5_banking_lora
    from google.colab import files
    files.download('qwen3.5_banking_lora.zip')
else:
    print(f"Adapter saved locally at: {os.path.abspath('qwen3.5_banking_lora')}")
    print("You can manually zip and transfer this folder.")